# Density-Based and Centroid-Based Clustering for Time Series Anomaly Detection

**Author:** Sawood Anwar 
**ORCID:** 0009-0000-2819-9179 
**GitHub:** [sawoodanwar](https://github.com/sawoodanwar)

---

This notebook demonstrates two unsupervised clustering approaches for detecting anomalous and low-quality patterns in time series data:

- **DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) — identifies dense regions and flags sparse points as outliers/anomalies without requiring a pre-specified number of clusters.
- **K-Means** — partitions observations into *k* groups; points far from any centroid are candidates for anomalous or inconsistent data quality.

The dataset used is the [Air Quality UCI dataset](https://archive.ics.uci.edu/ml/datasets/Air+Quality) — a real-world, publicly available multivariate time series with known missing values and sensor drift, making it a suitable proxy for industrial data quality analysis.

The workflow mirrors the core task described in Euronext Clearing's research position (Ref. R28837): investigating how unsupervised ML techniques can support automated detection of anomalous, inconsistent, or low-quality patterns in continuous data streams.

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.stats import zscore

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded.')

## 2. Load and Prepare Data

The Air Quality dataset contains hourly averaged responses from an array of metal oxide chemical sensors embedded in an air quality chemical multisensor device. It includes 9,358 instances with known missing values tagged as -200.

In [ ]:
# Load from UCI repository (semicolon-separated, European decimal comma)
url = (
    'https://archive.ics.uci.edu/ml/machine-learning-databases/'
    '00360/AirQualityUCI.zip'
)

# Download and read
df_raw = pd.read_csv(
    url,
    compression='zip',
    sep=';',
    decimal=',',
    parse_dates={'datetime': ['Date', 'Time']},
    dayfirst=True,
    na_values=-200
)

# Drop fully empty trailing columns
df_raw = df_raw.dropna(axis=1, how='all')
df_raw = df_raw.set_index('datetime').sort_index()

print(f'Shape: {df_raw.shape}')
print(f'Date range: {df_raw.index.min()} → {df_raw.index.max()}')
df_raw.head()

In [ ]:
# Select sensor columns with sufficient coverage
FEATURES = ['CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)',
            'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)',
            'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']

df = df_raw[FEATURES].copy()

# Report missingness before imputation
missing_pct = (df.isna().sum() / len(df) * 100).round(2)
print('Missing values (%) per feature:')
print(missing_pct.to_string())

# Forward-fill then back-fill short gaps (mimics sensor dropout handling)
df = df.ffill(limit=3).bfill(limit=3)

# Drop rows where imputation was insufficient
df = df.dropna()
print(f'\nRows after cleaning: {len(df)}')

## 3. Feature Engineering

For clustering we construct a **sliding-window feature matrix**: each row represents a 24-hour window summarised by its mean, standard deviation, and rate-of-change (first-order difference mean) for each sensor. This captures both level and volatility — key signals for data quality anomalies such as stuck sensors, spikes, and structural breaks.

In [ ]:
WINDOW = 24  # hours

def rolling_features(df, window):
    """Compute rolling mean, std, and mean abs diff for each column."""
    means = df.rolling(window).mean().add_suffix('_mean')
    stds = df.rolling(window).std().add_suffix('_std')
    diffs = df.diff().abs().rolling(window).mean().add_suffix('_roc')
    return pd.concat([means, stds, diffs], axis=1).dropna()

feat_df = rolling_features(df, WINDOW)
print(f'Feature matrix shape: {feat_df.shape}')
feat_df.head(3)

In [ ]:
# Standardise — required for both DBSCAN (distance-based) and K-Means
scaler = StandardScaler()
X = scaler.fit_transform(feat_df)

# Reduce to 2D via PCA for visualisation (retain components for later)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
print(f'Variance explained by 2 PCs: {pca.explained_variance_ratio_.sum():.1%}')

## 4. DBSCAN — Density-Based Anomaly Detection

DBSCAN groups observations in dense regions and assigns **label = -1** to noise points — observations that do not belong to any cluster. These noise points serve as anomaly flags.

Key hyperparameters:
- `eps` — the neighbourhood radius. Selected via the k-distance elbow plot.
- `min_samples` — minimum points to form a dense region.

In [ ]:
# k-distance plot to guide eps selection
from sklearn.neighbors import NearestNeighbors

k = 5
nbrs = NearestNeighbors(n_neighbors=k).fit(X)
distances, _ = nbrs.kneighbors(X)
k_distances = np.sort(distances[:, k-1])[::-1]

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(k_distances, linewidth=1.2)
ax.set_xlabel('Points (sorted by distance)', fontsize=11)
ax.set_ylabel(f'{k}-NN distance', fontsize=11)
ax.set_title('k-Distance Plot — Elbow indicates optimal eps', fontsize=12)
ax.axhline(y=3.5, color='crimson', linestyle='--', label='Selected eps = 3.5')
ax.legend()
plt.tight_layout()
plt.savefig('k_distance_plot.png', bbox_inches='tight')
plt.show()

In [ ]:
# Fit DBSCAN
dbscan = DBSCAN(eps=3.5, min_samples=5, metric='euclidean', n_jobs=-1)
db_labels = dbscan.fit_predict(X)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()
noise_pct = n_noise / len(db_labels) * 100

print(f'DBSCAN results:')
print(f'  Clusters found : {n_clusters}')
print(f'  Noise points   : {n_noise} ({noise_pct:.1f}% — flagged as anomalous)')
print(f'  Cluster sizes  : {pd.Series(db_labels[db_labels != -1]).value_counts().to_dict()}')

In [ ]:
# Visualise DBSCAN output in PCA space
fig, ax = plt.subplots(figsize=(9, 5))

palette = sns.color_palette('tab10', n_clusters)
for i, label in enumerate(sorted(set(db_labels))):
    mask = db_labels == label
    if label == -1:
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c='crimson', s=10, alpha=0.6, label='Anomaly (noise)', zorder=3)
    else:
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   s=8, alpha=0.4, label=f'Cluster {label}')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
ax.set_title('DBSCAN Clustering (PCA projection) — Anomalies in red', fontsize=12)
ax.legend(markerscale=2, fontsize=9)
plt.tight_layout()
plt.savefig('dbscan_pca.png', bbox_inches='tight')
plt.show()

In [ ]:
# Plot anomalies on the original time axis
result_df = feat_df.copy()
result_df['dbscan_label'] = db_labels
result_df['anomaly_dbscan'] = db_labels == -1

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(result_df.index, result_df['CO(GT)_mean'],
        linewidth=0.8, color='steelblue', label='CO(GT) 24h mean')
anomalies = result_df[result_df['anomaly_dbscan']]
ax.scatter(anomalies.index, anomalies['CO(GT)_mean'],
           color='crimson', s=15, zorder=5, label=f'DBSCAN anomaly ({len(anomalies)} points)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('CO(GT) mean', fontsize=11)
ax.set_title('DBSCAN Anomaly Detection on CO Sensor Time Series', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('dbscan_timeseries.png', bbox_inches='tight')
plt.show()

## 5. K-Means Clustering

K-Means partitions observations into *k* groups by minimising within-cluster variance. Points with high distance to their assigned centroid indicate anomalous or low-quality data patterns.

We select *k* using the **elbow method** (inertia) and validate with the **Silhouette score** and **Davies-Bouldin index**.

In [ ]:
# Elbow and Silhouette analysis
K_RANGE = range(2, 11)
inertias, silhouettes, db_scores = [], [], []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels, sample_size=2000, random_state=42))
    db_scores.append(davies_bouldin_score(X, labels))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(list(K_RANGE), inertias, 'o-', color='steelblue')
axes[0].set_title('Elbow Method (Inertia)', fontsize=11)
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')

axes[1].plot(list(K_RANGE), silhouettes, 'o-', color='seagreen')
axes[1].set_title('Silhouette Score (higher = better)', fontsize=11)
axes[1].set_xlabel('k')
axes[1].set_ylabel('Score')

axes[2].plot(list(K_RANGE), db_scores, 'o-', color='darkorange')
axes[2].set_title('Davies-Bouldin Index (lower = better)', fontsize=11)
axes[2].set_xlabel('k')
axes[2].set_ylabel('Score')

plt.suptitle('K-Means Model Selection', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('kmeans_selection.png', bbox_inches='tight')
plt.show()

best_k = list(K_RANGE)[np.argmax(silhouettes)]
print(f'Best k by Silhouette: {best_k}')

In [ ]:
# Fit final K-Means
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
km_labels = km_final.fit_predict(X)

# Distance to centroid as anomaly score
centroid_distances = np.min(
    np.linalg.norm(X[:, np.newaxis] - km_final.cluster_centers_, axis=2), axis=1
)

# Flag top 5% by distance as anomalous
threshold = np.percentile(centroid_distances, 95)
result_df['kmeans_label'] = km_labels
result_df['centroid_dist'] = centroid_distances
result_df['anomaly_kmeans'] = centroid_distances > threshold

print(f'K-Means: {best_k} clusters')
print(f'Anomaly threshold (95th pct): {threshold:.3f}')
print(f'Flagged anomalies: {result_df["anomaly_kmeans"].sum()}')

In [ ]:
# Visualise K-Means in PCA space
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: cluster membership
scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1],
                          c=km_labels, cmap='tab10', s=8, alpha=0.5)
centroids_2d = pca.transform(km_final.cluster_centers_)
axes[0].scatter(centroids_2d[:, 0], centroids_2d[:, 1],
                c='black', marker='X', s=120, zorder=5, label='Centroids')
axes[0].set_title(f'K-Means Clusters (k={best_k})', fontsize=11)
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(fontsize=9)

# Right: anomaly overlay
normal = result_df[~result_df['anomaly_kmeans']]
anomalous = result_df[ result_df['anomaly_kmeans']]
normal_idx = feat_df.index.get_indexer(normal.index)
anomalous_idx = feat_df.index.get_indexer(anomalous.index)
axes[1].scatter(X_pca[normal_idx, 0], X_pca[normal_idx, 1],
                s=6, alpha=0.3, color='steelblue', label='Normal')
axes[1].scatter(X_pca[anomalous_idx, 0], X_pca[anomalous_idx, 1],
                s=12, alpha=0.8, color='crimson', label='Anomaly (top 5%)')
axes[1].set_title('K-Means Anomalies (centroid distance > 95th pct)', fontsize=11)
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].legend(markerscale=2, fontsize=9)

plt.tight_layout()
plt.savefig('kmeans_pca.png', bbox_inches='tight')
plt.show()

## 6. Comparative Evaluation: DBSCAN vs K-Means

A key objective in the Euronext Clearing role is comparative assessment of alternative approaches. Here we compare the two methods on anomaly agreement, temporal distribution, and interpretability.

In [ ]:
# Agreement between methods
both = (result_df['anomaly_dbscan'] & result_df['anomaly_kmeans']).sum()
only_d = (result_df['anomaly_dbscan'] & ~result_df['anomaly_kmeans']).sum()
only_k = (~result_df['anomaly_dbscan'] & result_df['anomaly_kmeans']).sum()
neither = (~result_df['anomaly_dbscan'] & ~result_df['anomaly_kmeans']).sum()

agreement = pd.DataFrame({
    'Category': ['Both flag anomaly', 'DBSCAN only', 'K-Means only', 'Neither'],
    'Count': [both, only_d, only_k, neither],
    'Pct (%)': [round(x / len(result_df) * 100, 1) for x in [both, only_d, only_k, neither]]
})
print(agreement.to_string(index=False))

In [ ]:
# Temporal distribution of anomalies
result_df['month'] = result_df.index.to_period('M')
monthly = result_df.groupby('month')[['anomaly_dbscan', 'anomaly_kmeans']].sum()
monthly.index = monthly.index.astype(str)

fig, ax = plt.subplots(figsize=(13, 4))
x = np.arange(len(monthly))
w = 0.35
ax.bar(x - w/2, monthly['anomaly_dbscan'], w, label='DBSCAN anomalies', color='steelblue', alpha=0.8)
ax.bar(x + w/2, monthly['anomaly_kmeans'], w, label='K-Means anomalies', color='darkorange', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(monthly.index, rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Anomaly count', fontsize=11)
ax.set_title('Monthly Anomaly Counts: DBSCAN vs K-Means', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('comparison_monthly.png', bbox_inches='tight')
plt.show()

In [ ]:
# Summary comparison table
summary = pd.DataFrame({
    'Method': ['DBSCAN', 'K-Means (centroid distance)'],
    'Hyperparameters': ['eps=3.5, min_samples=5', f'k={best_k}, threshold=95th pct'],
    'Anomalies flagged': [result_df['anomaly_dbscan'].sum(), result_df['anomaly_kmeans'].sum()],
    'Pct of data': [f"{result_df['anomaly_dbscan'].mean()*100:.1f}%",
                    f"{result_df['anomaly_kmeans'].mean()*100:.1f}%"],
    'Requires k upfront': ['No', 'Yes'],
    'Handles varying density': ['Yes', 'No'],
    'Interpretability': ['Noise = structural outlier', 'Distance = deviation from typical pattern'],
})
print(summary.to_string(index=False))

## 7. Discussion

**DBSCAN** is parameter-sensitive but requires no pre-specified cluster count and naturally handles irregular cluster shapes. Its noise points (label = -1) correspond directly to structurally anomalous observations — analogous to missing value clusters, sensor failures, or data pipeline errors in a financial clearing context.

**K-Means** is computationally efficient and produces an interpretable centroid-distance anomaly score, enabling a continuous risk ranking rather than a binary flag. However, it assumes roughly spherical clusters and requires *k* to be specified in advance.

**Key finding:** the two methods agree on a subset of high-confidence anomalies. Observations flagged by both are the highest-priority candidates for data quality investigation. Method-specific flags likely capture different failure modes — structural discontinuities (DBSCAN) vs. statistical extremes (K-Means) — suggesting an ensemble approach as a productive direction for further research.

This comparative framework directly maps onto the Euronext Clearing research objective: designing and evaluating alternative cluster formation methodologies to identify anomalous, inconsistent, or low-quality data patterns in large-scale time series datasets.

In [ ]:
# Export flagged anomalies for downstream review
export = result_df[result_df['anomaly_dbscan'] | result_df['anomaly_kmeans']][
    ['dbscan_label', 'anomaly_dbscan', 'kmeans_label', 'centroid_dist', 'anomaly_kmeans']
].copy()
export['flagged_by_both'] = export['anomaly_dbscan'] & export['anomaly_kmeans']
export.to_csv('anomaly_flags.csv')
print(f'Exported {len(export)} flagged records to anomaly_flags.csv')